# Math LLM Probing & Evaluation Pipeline

This notebook implements a pipeline to:
1. Load `Qwen/Qwen2.5-Math-7B-Instruct`.
2. Generate answers to math problems (enforcing direct answers).
3. Extract hidden states.
4. Evaluate answers using Regex and SymPy.

In [ ]:
import os
import sys

# 1. Clone the repo if it doesn't exist
REPO_NAME = "probing-llm-math"
if not os.path.exists(REPO_NAME):
    print("Cloning repository...")
    !git clone -b colab-pipeline-setup https://github.com/1hamzaiqbal/probing-llm-math.git
else:
    print(f"{REPO_NAME} already exists.")

# 2. Install dependencies
print("Installing dependencies...")
!pip install -q -r {REPO_NAME}/requirements.txt

# 3. Add to path so we can import src
if REPO_NAME not in sys.path:
    sys.path.append(REPO_NAME)

print("Setup complete!")

In [ ]:
try:
    from src import model_utils, evaluator
    print("Successfully imported modules from src.")
except ImportError as e:
    print(f"Error importing modules: {e}")
    print("Current path:", sys.path)
    print("Current dir contents:", os.listdir("."))
    if os.path.exists(REPO_NAME):
        print(f"{REPO_NAME} contents:", os.listdir(REPO_NAME))

In [ ]:
# Load Model
model, tokenizer = model_utils.load_model(model_name="Qwen/Qwen2.5-Math-7B-Instruct")

In [ ]:
# Define some test questions (Hendrycks MATH / AIME style)
questions = [
    # Simple Arithmetic
    {"question": "What is 2 + 2?", "answer": "4"},
    
    # Algebra (Implicit multiplication check)
    {"question": "Calculate the derivative of x^2", "answer": "2x"},
    
    # Geometry (Distance formula, integer result)
    {"question": "Find the distance between the points (2, -5) and (-4, 3).", "answer": "10"},
    
    # Algebra (Roots of quadratic)
    {"question": "Find the sum of the roots of x^2 - 5x + 6 = 0.", "answer": "5"},
    
    # Fractions
    {"question": "Simplify 1/2 + 1/3.", "answer": "5/6"},
    
    # Trigonometry (LaTeX check)
    {"question": "What is the value of \\sin(\\pi/2)?", "answer": "1"},
    
    # Radicals (SymPy check)
    {"question": "Simplify \\sqrt{8}.", "answer": "2\\sqrt{2}"},
    
    # AIME Style (Integer answer)
    {"question": "Let $S$ be the set of all positive integers $n$ such that $n^2$ is a multiple of 72. What is the smallest element of $S$?", "answer": "12"}
]

In [ ]:
# Run Pipeline
results = []

for item in questions:
    q = item["question"]
    gt = item["answer"]
    
    print(f"Processing: {q}")
    
    # Generate
    pred_text, hidden_states = model_utils.generate_answer(model, tokenizer, q)
    
    # Extract clean answer
    clean_pred = evaluator.extract_answer(pred_text)
    
    # Evaluate
    is_correct = evaluator.is_equivalent(clean_pred, gt)
    
    print(f"  Pred: {pred_text}")
    print(f"  Clean: {clean_pred}")
    print(f"  Correct: {is_correct}")
    
    results.append({
        "question": q,
        "ground_truth": gt,
        "prediction": pred_text,
        "clean_prediction": clean_pred,
        "correct": is_correct,
        # Store hidden states if needed, or save to disk to save RAM
        # "hidden_states": hidden_states 
    })

In [ ]:
# Analyze Results
correct_count = sum(1 for r in results if r['correct'])
accuracy = correct_count / len(results)
print(f"Accuracy: {accuracy:.2%}")